# Win Pipeline: LightGBM + XGBoost + CatBoost -> OOF Stacking
Ensemble of 3 diverse models with seed-averaged 5-fold CV, plus a stacked meta-model. Picks whichever strategy scores best on OOF AUC and writes `submission.csv`.

In [ ]:
!pip install lightgbm xgboost catboost -q

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

SEED = 42
N_FOLDS = 5
SEEDS_FOR_BAGGING = [42, 202, 777]   # seed-averaging for extra stability

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Feature Engineering

In [ ]:
num_cols = ['age','daily_screen_time_hours','social_media_hours','gaming_hours',
            'work_study_hours','sleep_hours','notifications_per_day',
            'app_opens_per_day','weekend_screen_time']
cat_cols = ['gender','stress_level','academic_work_impact']

# Ordinal map for stress_level -- adjust labels if your actual categories differ
stress_map = {'Low': 0, 'Medium': 1, 'High': 2}

for df in [train, test]:
    df['screen_to_sleep_ratio']    = df['daily_screen_time_hours'] / (df['sleep_hours'] + 1e-3)
    df['social_share_of_screen']   = df['social_media_hours'] / (df['daily_screen_time_hours'] + 1e-3)
    df['weekend_vs_weekday']       = df['weekend_screen_time'] - df['daily_screen_time_hours']
    df['leisure_screen_time']      = df['social_media_hours'] + df['gaming_hours']
    df['opens_per_notification']   = df['app_opens_per_day'] / (df['notifications_per_day'] + 1e-3)
    df['notifications_per_open']   = df['notifications_per_day'] / (df['app_opens_per_day'] + 1e-3)
    df['work_leisure_ratio']       = df['work_study_hours'] / (df['leisure_screen_time'] + 1e-3)
    df['screen_time_gap']          = df['weekend_screen_time'] - df['work_study_hours']
    df['sleep_per_age']            = df['sleep_hours'] / (df['age'] + 1e-3)
    df['gaming_share_of_screen']   = df['gaming_hours'] / (df['daily_screen_time_hours'] + 1e-3)

    # Ordinal version of stress_level (numeric, keeps the natural ordering)
    df['stress_level_ord'] = df['stress_level'].map(stress_map)

    # Interaction feature
    df['stress_x_social'] = df['stress_level_ord'] * df['social_media_hours']

new_num_cols = num_cols + [
    'screen_to_sleep_ratio', 'social_share_of_screen', 'weekend_vs_weekday',
    'leisure_screen_time', 'opens_per_notification', 'notifications_per_open',
    'work_leisure_ratio', 'screen_time_gap', 'sleep_per_age',
    'gaming_share_of_screen', 'stress_level_ord', 'stress_x_social'
]

for c in cat_cols:
    train[c] = train[c].fillna('Missing').astype('category')
    test[c] = test[c].fillna('Missing').astype('category')

features = new_num_cols + cat_cols
X = train[features].copy()
y = train['addicted_label']
X_test = test[features].copy()

cat_feature_idx = [X.columns.get_loc(c) for c in cat_cols]  # for CatBoost


## 2. Base Model Functions (OOF generation, seed-averaged)

In [ ]:
def run_lgbm(X, y, X_test, cat_cols):
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    params = dict(objective='binary', metric='auc', n_estimators=500,
                   learning_rate=0.02, num_leaves=48, min_child_samples=30,
                   subsample=0.8, colsample_bytree=0.8,
                   reg_alpha=0.5, reg_lambda=0.5, verbosity=-1)
    for seed in SEEDS_FOR_BAGGING:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for tr_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
            model = lgb.LGBMClassifier(**{**params, 'random_state': seed})
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                      callbacks=[lgb.early_stopping(150, verbose=False)])
            oof[val_idx] += model.predict_proba(X_val)[:, 1] / len(SEEDS_FOR_BAGGING)
            test_pred += model.predict_proba(X_test)[:, 1] / (N_FOLDS * len(SEEDS_FOR_BAGGING))
    return oof, test_pred

def run_xgb(X, y, X_test, cat_cols):
    X_enc = pd.get_dummies(X, columns=cat_cols)
    X_test_enc = pd.get_dummies(X_test, columns=cat_cols)
    X_enc, X_test_enc = X_enc.align(X_test_enc, join='left', axis=1, fill_value=0)

    oof = np.zeros(len(X_enc))
    test_pred = np.zeros(len(X_test_enc))
    params = dict(objective='binary:logistic', eval_metric='auc', n_estimators=3000,
                   learning_rate=0.02, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                   reg_alpha=0.5, reg_lambda=1.0, tree_method='hist')
    for seed in SEEDS_FOR_BAGGING:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for tr_idx, val_idx in skf.split(X_enc, y):
            X_tr, X_val = X_enc.iloc[tr_idx], X_enc.iloc[val_idx]
            y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
            model = xgb.XGBClassifier(**{**params, 'random_state': seed, 'early_stopping_rounds': 150})
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            oof[val_idx] += model.predict_proba(X_val)[:, 1] / len(SEEDS_FOR_BAGGING)
            test_pred += model.predict_proba(X_test_enc)[:, 1] / (N_FOLDS * len(SEEDS_FOR_BAGGING))
    return oof, test_pred

def run_catboost(X, y, X_test, cat_feature_idx):
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    for seed in SEEDS_FOR_BAGGING:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for tr_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
            model = cb.CatBoostClassifier(
                iterations=3000, learning_rate=0.02, depth=6,
                loss_function='Logloss', eval_metric='AUC',
                random_seed=seed, verbose=False, early_stopping_rounds=150,
                cat_features=cat_feature_idx
            )
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
            oof[val_idx] += model.predict_proba(X_val)[:, 1] / len(SEEDS_FOR_BAGGING)
            test_pred += model.predict_proba(X_test)[:, 1] / (N_FOLDS * len(SEEDS_FOR_BAGGING))
    return oof, test_pred


## 3. Train All Three Models
This is the slow part: 3 models x 3 seeds x 5 folds. If you just want to sanity-check the code runs, temporarily set `SEEDS_FOR_BAGGING = [42]` in the first cell, then scale back up for your real submission.

In [ ]:
print("Training LightGBM...")
oof_lgb, test_lgb = run_lgbm(X, y, X_test, cat_cols)
print("LGBM OOF AUC:", roc_auc_score(y, oof_lgb))

print("Training XGBoost...")
oof_xgb, test_xgb = run_xgb(X, y, X_test, cat_cols)
print("XGB OOF AUC:", roc_auc_score(y, oof_xgb))

print("Training CatBoost...")
oof_cb, test_cb = run_catboost(X, y, X_test, cat_feature_idx)
print("CatBoost OOF AUC:", roc_auc_score(y, oof_cb))


Training LightGBM...


C:\Users\Admin\AppData\Roaming\Python\Python314\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\Admin\AppData\Roaming\Python\Python314\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\Admin\AppData\Roaming\Python\Python314\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\Admin\AppData\Roaming\Python\Python314\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  ev

KeyboardInterrupt: 

## 4. Simple Average Blend

In [ ]:
oof_avg = (oof_lgb + oof_xgb + oof_cb) / 3
test_avg = (test_lgb + test_xgb + test_cb) / 3
print("Simple average blend OOF AUC:", roc_auc_score(y, oof_avg))


## 5. Stacked Meta-Model (Logistic Regression on OOF predictions)

In [ ]:
stack_X = np.column_stack([oof_lgb, oof_xgb, oof_cb])
stack_X_test = np.column_stack([test_lgb, test_xgb, test_cb])

meta = LogisticRegression()
meta.fit(stack_X, y)

stack_oof_pred = meta.predict_proba(stack_X)[:, 1]
stack_test_pred = meta.predict_proba(stack_X_test)[:, 1]
print("Stacked meta-model OOF AUC:", roc_auc_score(y, stack_oof_pred))
print("Meta-model weights (lgb, xgb, cb):", meta.coef_)


## 6. Pick the Best Strategy and Write submission.csv

In [ ]:
scores = {
    'lgbm_only': (roc_auc_score(y, oof_lgb), test_lgb),
    'simple_avg': (roc_auc_score(y, oof_avg), test_avg),
    'stacked': (roc_auc_score(y, stack_oof_pred), stack_test_pred),
}
best_name = max(scores, key=lambda k: scores[k][0])
best_auc, best_test_pred = scores[best_name]
print(f"Best strategy: {best_name} (OOF AUC {best_auc:.5f})")

submission = pd.DataFrame({'id': test['id'], 'addicted_label': best_test_pred})
submission.to_csv('submission.csv', index=False)
submission.head()
